In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

def gerar_holograma_titanic(caminho_csv):
    # =========================================================================
    # 1. PIPELINE DE DADOS: LEITURA E PROCESSAMENTO
    # =========================================================================
    df = pd.read_csv(caminho_csv)

    # Tratamento de tipagem (converte vírgula para ponto caso o CSV seja BR)
    if df['Age'].dtype == object:
        df['Age'] = df['Age'].str.replace(',', '.').astype(float)

    df['SibSp'] = df['SibSp'].fillna(0)
    df['Parch'] = df['Parch'].fillna(0)
    df['FamilySize'] = df['SibSp'] + df['Parch']

    # Separação binária do target
    sobreviventes = df[df['Survived'] == 1]
    mortos = df[df['Survived'] == 0]

    # Função construtora do perfil geométrico (Proporção % vs Volume Absoluto)
    def extrair_perfil(subset):
        t = len(subset) if len(subset) > 0 else 1 # Trava de divisão por zero
        return {
            '1ª Classe': {'pct': (len(subset[subset['Pclass']==1])/t)*100, 'abs': len(subset[subset['Pclass']==1])},
            '3ª Classe': {'pct': (len(subset[subset['Pclass']==3])/t)*100, 'abs': len(subset[subset['Pclass']==3])},
            'Com Família': {'pct': (len(subset[subset['FamilySize']>0])/t)*100, 'abs': len(subset[subset['FamilySize']>0])},
            'Viajava Sozinho': {'pct': (len(subset[subset['FamilySize']==0])/t)*100, 'abs': len(subset[subset['FamilySize']==0])},
            'Embarque S.': {'pct': (len(subset[subset['Embarked']=='S'])/t)*100, 'abs': len(subset[subset['Embarked']=='S'])},
            'Jovens (18-35)': {'pct': (len(subset[(subset['Age']>=18) & (subset['Age']<=35)])/t)*100, 'abs': len(subset[(subset['Age']>=18) & (subset['Age']<=35)])},
        }

    perfil_sob = extrair_perfil(sobreviventes)
    perfil_mor = extrair_perfil(mortos)

    categorias = list(perfil_sob.keys())
    N = len(categorias)
    angulos = np.linspace(0, 2 * np.pi, N, endpoint=False)

    fig = go.Figure()

    # =========================================================================
    # 2. MOTOR GEOMÉTRICO (ESCALAS E CHÃO)
    # =========================================================================
    # Fatores para normalizar dados em coordenadas Cartesianas visuais agradáveis
    fator_r = 10 / 100.0  # O raio máximo da teia será 10 (representando 100%)
    fator_z = 15 / 200.0  # O teto visual será 15 (200 pessoas representam o pico máximo dos dados)

    # Teia do chão (Círculos concêntricos a cada 20%)
    for pct in [20, 40, 60, 80, 100]:
        r = pct * fator_r
        theta_grid = np.linspace(0, 2 * np.pi, 100)
        fig.add_trace(go.Scatter3d(
            x=r * np.cos(theta_grid), y=r * np.sin(theta_grid), z=np.zeros(100),
            mode='lines', line=dict(color='#cccccc' if pct != 100 else '#505050', width=1.5 if pct == 100 else 1),
            hoverinfo='skip', showlegend=False
        ))
        fig.add_trace(go.Scatter3d(
            x=[r], y=[0], z=[0.5], mode='text', text=[f"{pct}%"],
            textfont=dict(color='#808080', size=10), hoverinfo='skip', showlegend=False
        ))

    # Raios polares (Divisórias das categorias)
    for angulo in angulos:
        fig.add_trace(go.Scatter3d(
            x=[0, 10 * np.cos(angulo)], y=[0, 10 * np.sin(angulo)], z=[0, 0],
            mode='lines', line=dict(color='#cccccc', width=1), hoverinfo='skip', showlegend=False
        ))

    # =========================================================================
    # 3. RENDERIZAÇÃO DAS MALHAS 3D
    # =========================================================================
    def plotar_malha(dados_perfil, cor_hex, cor_rgba, nome):
        pcts = [dados_perfil[cat]['pct'] for cat in categorias]
        abss = [dados_perfil[cat]['abs'] for cat in categorias]

        # Coordenadas espaciais dos vértices
        raios_grafico = np.array(pcts) * fator_r
        x_data = raios_grafico * np.cos(angulos)
        y_data = raios_grafico * np.sin(angulos)
        z_real = np.array(abss)
        z_plot = z_real * fator_z

        # Matrizes fechadas para o laço da linha
        x_f = np.append(x_data, x_data[0])
        y_f = np.append(y_data, y_data[0])
        z_plot_f = np.append(z_plot, z_plot[0])
        z_real_f = np.append(z_real, z_real[0])
        pcts_f = np.append(pcts, pcts[0])
        cat_f = categorias + [categorias[0]]

        # Renderiza o Corpo Translúcido
        x_mesh, y_mesh, z_mesh = np.concatenate(([0], x_data)), np.concatenate(([0], y_data)), np.concatenate(([0], z_plot))
        fig.add_trace(go.Mesh3d(
            x=x_mesh, y=y_mesh, z=z_mesh,
            i=[0]*N, j=list(range(1, N + 1)), k=list(range(2, N + 1)) + [1],
            color=cor_rgba, opacity=1, hoverinfo='skip', name=f'Volume {nome}'
        ))

        # Renderiza o Exoesqueleto (Linhas e Picos)
        fig.add_trace(go.Scatter3d(
            x=x_f, y=y_f, z=z_plot_f, mode='lines+markers',
            line=dict(color=cor_hex, width=3),
            marker=dict(size=4, color=cor_hex, symbol='circle'),
            customdata=np.stack((cat_f, pcts_f, z_real_f), axis=-1),
            hovertemplate=(
                f"<b>Grupo: {nome}</b><br><br>"
                "<b>%{customdata[0]}</b><br>"
                "Proporção do Perfil: %{customdata[1]:.1f}%<br>"
                "Volume Absoluto (Vidas): %{customdata[2]}<extra></extra>"
            ),
            name=nome
        ))

        # Drop Lines (Linhas âncora)
        for i, ang in enumerate(angulos):
            fig.add_trace(go.Scatter3d(
                x=[x_data[i], x_data[i]], y=[y_data[i], y_data[i]], z=[0, z_plot[i]],
                mode='lines', line=dict(color=cor_hex, width=2, dash='dot'), hoverinfo='skip', showlegend=False
            ))

    # Plota a malha dos mortos (Vermelho Alerta) e sobreviventes (Verde Água)
    plotar_malha(perfil_mor, '#e74c3c', 'rgba(231, 76, 60, 0.25)', 'Não Sobreviveram')
    plotar_malha(perfil_sob, '#1abc9c', 'rgba(26, 188, 156, 0.45)', 'Sobreviveram')

    # =========================================================================
    # 4. COMPONENTES ESTÁTICOS (EIXO Z E BILLBOARDING)
    # =========================================================================
    fig.add_trace(go.Scatter3d(
        x=[0, 0], y=[0, 0], z=[0, 15], mode='lines',
        line=dict(color='#000000', width=0.5), hoverinfo='skip', showlegend=False
    ))
    fig.add_trace(go.Scatter3d(
        x=[0, 0, 0], y=[0, 0, 0], z=[0, 15], mode='text',
        text=['',' 200 pessoas'], textposition='middle right',
        textfont=dict(color='#000000', size=11), hoverinfo='skip', showlegend=False
    ))

    anotacoes = [
        dict(showarrow=False, x=13 * np.cos(a), y=13 * np.sin(a), z=0, text=c.upper(),
             font=dict(color='#000000', size=11, family="Arial", weight="bold"),
             xanchor='center', yanchor='middle')
        for a, c in zip(angulos, categorias)
    ]

    # =========================================================================
    # 5. CÂMERA E TEMA (UI/UX)
    # =========================================================================
    fig.update_layout(
        title=dict(
            text="Holograma Comparativo: Sobreviventes vs Vítimas",
            font=dict(size=22, color='#ffffff'), x=0.5, y=0.9
        ),
        scene=dict(
            xaxis=dict(visible=False, range=[-15, 15]),
            yaxis=dict(visible=False, range=[-15, 15]),
            zaxis=dict(visible=False, range=[-2, 20]),
            aspectratio=dict(x=1, y=1, z=0.55),
            annotations=anotacoes,
            camera=dict(
                up=dict(x=0, y=0, z=1), center=dict(x=0, y=0, z=-0.1),
                eye=dict(x=1.5, y=-1.5, z=0.8) # Posição baixa para enfatizar a "montanha" de mortes
            ),
            bgcolor='#ffffff'
        ),
        paper_bgcolor='#1F2530',
        margin=dict(l=0, r=0, t=80, b=0),
        showlegend=True,
        legend=dict(yanchor="top", y=0.85, xanchor="left", x=0.05, font=dict(color="white"), bgcolor="rgba(0,0,0,0)"),
        width=950, height=850
    )

    fig.show()

# EXECUÇÃO DA CÉLULA (Basta chamar passando o nome exato do seu arquivo no Colab)
gerar_holograma_titanic('GTH - tested.csv')

In [ ]:
import plotly.express as px
import pandas as pd

df = pd.read_csv('GTH - tested.csv')

# Limpeza e tipagem matemática dos eixos contínuos
if df['Age'].dtype == object: df['Age'] = df['Age'].str.replace(',', '.').astype(float)
if df['Fare'].dtype == object: df['Fare'] = df['Fare'].str.replace(',', '.').astype(float)

df['Idade'] = df['Age'].fillna(df['Age'].mean()) # Imputação de dados faltantes com a média
df['Tamanho_Familia'] = df['SibSp'] + df['Parch'] + 1
df['Status_Sobrevivencia'] = df['Survived'].map({1: 'Sobreviveu', 0: 'Não Sobreviveu'})

fig = px.scatter_3d(
    df,
    x='Idade',
    y='Fare',
    z='Pclass',
    color='Status_Sobrevivencia',
    size='Tamanho_Familia', # O volume da bolha indica quantas pessoas viajavam junto
    hover_name='Name',
    color_discrete_map={'Sobreviveu': '#2ECC71', 'Não Sobreviveu': '#E74C3C'},
    title="Micro-Análise: Posição de Cada Passageiro no Navio",
    labels={'Fare': 'Tarifa Paga ($)', 'Pclass': 'Classe Social'}
)

fig.update_layout(
    scene=dict(
        xaxis=dict(backgroundcolor="#2C3E50", gridcolor="white", showbackground=True),
        yaxis=dict(backgroundcolor="#34495E", gridcolor="white", showbackground=True),
        zaxis=dict(backgroundcolor="#2C3E50", gridcolor="white", showbackground=True, autorange="reversed"), # Classe 1 em cima
    ),
    paper_bgcolor='#1F2530', font_color='white', width=900, height=700
)
fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Mude para o nome exato do seu arquivo no Colab
arquivo_csv = 'GTH - tested.csv'
df = pd.read_csv(arquivo_csv)

# 1. Preparação dos Nós
labels = ["1ª Classe", "2ª Classe", "3ª Classe", "Mulheres", "Homens", "Sobreviveram", "Não Sobreviveram"]
cores_nos = ["#F39C12", "#E67E22", "#D35400", "#1ABC9C", "#34495E", "#2ECC71", "#E74C3C"]

sources, targets, values, link_colors = [], [], [], []

# 2. Algoritmo de Roteamento (CORRIGIDO PARA MASCULINO/FEMININO)
for pclass in [1, 2, 3]:
    for sex in ['Feminino', 'Masculino']:
        for surv in [1, 0]:
            count = len(df[(df['Pclass'] == pclass) & (df['Sex'] == sex) & (df['Survived'] == surv)])
            if count == 0: continue

            idx_classe = pclass - 1
            idx_sexo = 3 if sex == 'Feminino' else 4
            idx_surv = 5 if surv == 1 else 6

            # Fluxo A: Classe para Sexo
            sources.append(idx_classe)
            targets.append(idx_sexo)
            values.append(count)
            link_colors.append("rgba(26, 188, 156, 0.3)" if sex == 'Feminino' else "rgba(52, 73, 94, 0.3)")

            # Fluxo B: Sexo para Sobrevivência
            sources.append(idx_sexo)
            targets.append(idx_surv)
            values.append(count)
            link_colors.append("rgba(46, 204, 113, 0.4)" if surv == 1 else "rgba(231, 76, 60, 0.4)")

# 3. Renderização
fig = go.Figure(data=[go.Sankey(
    node=dict(pad=25, thickness=20, line=dict(color="black", width=0.5), label=labels, color=cores_nos),
    link=dict(source=sources, target=targets, value=values, color=link_colors)
)])

fig.update_layout(
    title_text="Fluxo de Vidas: Da Classe Social ao Destino Final",
    font_size=12, paper_bgcolor='#1F2530', font_color='white', width=900, height=600
)
fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

arquivo_csv = 'GTH - tested.csv'
df = pd.read_csv(arquivo_csv)

# Tratamento ETL (CORRIGIDO PARA MAPEAMENTO DO SEU CSV)
df['Status_Classe'] = df['Pclass'].map({1: '1ª Classe', 2: '2ª Classe', 3: '3ª Classe'})
df['Status_Sexo'] = df['Sex'].map({'Feminino': 'Mulheres', 'Masculino': 'Homens'})
df['Status_Sobrevivencia'] = df['Survived'].map({1: 'Sobreviveu', 0: 'Não Sobreviveu'})

# Agrupamento para contagem (Criando a raiz 'Navio')
df_agrupado = df.groupby(['Status_Sexo', 'Status_Classe', 'Status_Sobrevivencia']).size().reset_index(name='Contagem')
df_agrupado['Navio'] = 'Titanic (Total)'

# Renderização do Sunburst
fig = px.sunburst(
    df_agrupado,
    path=['Navio', 'Status_Sexo', 'Status_Classe', 'Status_Sobrevivencia'],
    values='Contagem',
    color='Status_Sobrevivencia',
    color_discrete_map={'Sobreviveu': '#1ABC9C', 'Não Sobreviveu': '#E74C3C', '(?!)': '#34495E'},
    title='Hierarquia Demográfica (Clique nas fatias para explorar)'
)

fig.update_layout(paper_bgcolor='#1F2530', font_color='white', width=800, height=800)
fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

# 1. Leitura e ETL (Tratamento das vírgulas nos decimais)
df = pd.read_csv('GTH - tested.csv')
if df['Fare'].dtype == object:
    df['Fare'] = df['Fare'].str.replace(',', '.').astype(float)

# Criação de labels amigáveis
df['Status'] = df['Survived'].map({1: 'Sobreviveu', 0: 'Não Sobreviveu'})

# 2. Renderização
fig = px.box(
    df,
    x='Status',
    y='Fare',
    color='Status',
    points='outliers', # Mostra apenas os pontos que fogem muito do padrão (outliers)
    color_discrete_map={'Sobreviveu': '#2ECC71', 'Não Sobreviveu': '#E74C3C'},
    labels={'Status': 'Destino', 'Fare': 'Tarifa Paga ($)'},
    title='Distribuição de Tarifas (Boxplot): Sobreviventes vs Vítimas'
)

fig.update_layout(
    paper_bgcolor='#1F2530',
    plot_bgcolor='#1F2530',
    font_color='white',
    width=800,
    height=600,
    # Adicionamos o 'range' recebendo [min, max]
    yaxis=dict(gridcolor='#34495E', zerolinecolor='#34495E', range=[0, 300])
)
fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

# 1. Leitura e ETL
df = pd.read_csv('GTH - tested.csv')

# Limpeza das colunas numéricas
if df['Age'].dtype == object: df['Age'] = df['Age'].str.replace(',', '.').astype(float)
if df['Fare'].dtype == object: df['Fare'] = df['Fare'].str.replace(',', '.').astype(float)

# Criando a variável FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch']

# 2. Seleção apenas de dados numéricos para a correlação
colunas_numericas = ['Survived', 'Pclass', 'Age', 'Fare', 'FamilySize']
df_corr = df[colunas_numericas].corr() # Calcula a matriz de correlação de Pearson

# 3. Renderização do Heatmap
fig = px.imshow(
    df_corr,
    text_auto='.2f', # Mostra a correlação com 2 casas decimais
    aspect="auto",
    color_continuous_scale='RdBu_r', # Escala: Azul para correlação positiva, Vermelho para negativa
    zmin=-1, zmax=1, # A correlação sempre vai de -1 a 1
    title='Matriz de Correlação de Pearson (Heatmap)'
)

fig.update_layout(paper_bgcolor='#1F2530', font_color='white', width=700, height=600)
fig.show()